# EWC — Proof of Concept

EWC adds a **regularisation penalty** to the loss function that resists changes to weights that were
important for previous tasks.

The total loss when training on task *k* is:

```
L(θ) = L_task(θ)  +  λ  ×  Σ_{t < k}  Σ_i  F_{t,i}  ×  (θ_i  −  θ*_{t,i})²
         ①           ②         ③              ④          ⑤          ⑥
```

---

### ① L_task(θ) — Task Loss

The standard Binary Cross-Entropy loss on the **current task's data**.
This is what the model learns from. Lower means the model fits the current task better.
What matters is that it *decreases* over training epochs.

> A value of 0.05 means the model's average prediction error is low.
> On its own the number is not very meaningful — the *trend* is what counts.

---

### ② λ (lambda) — Penalty Strength

A hyperparameter **you set**. Controls the trade-off between learning and remembering:

| λ too small | λ in the sweet spot | λ too large |
|:-----------:|:-------------------:|:-----------:|
| Penalty ignored → forgets | Learns new tasks, retains old ones | Penalty dominates → frozen, cannot learn |

---

### ③ Σ_{t < k} — Sum Over All Previous Tasks

The penalty accumulates over every task the model has already seen:

| Training on | Penalty sums over |
|:-----------:|:-----------------:|
| Task 1 | — (no penalty yet) |
| Task 2 | 1 previous task |
| Task 3 | 2 previous tasks |
| Task k | k − 1 previous tasks |

Even with the same λ, the total penalty **grows** as more tasks accumulate.
This is why EWC can become increasingly rigid over long task sequences.

---

### ④ F_{t,i} — Fisher Information

How **important** weight *i* was for task *t*. Computed after training on task *t* as the
average squared gradient:

```
F_{t,i} = E[ (∂L / ∂θ_i)² ]
```

High Fisher value → this weight strongly influenced the loss on task *t* → protect it.  
Low Fisher value → this weight barely mattered → allow it to change freely.

---

### ⑤ θ_i — Current Weight Value

The current value of weight *i* during training on task *k*.

---

### ⑥ θ*_{t,i} — Anchor Weight

The value weight *i* had **right after** training on task *t*. This is the anchor.
The penalty measures how far the weight has drifted from it since then.

---

### Putting it together: F_{t,i} × (θ_i − θ*_{t,i})²

| Situation | Penalty |
|-----------|--------|
| Important weight (high F) that has drifted far | Large → strongly resisted |
| Important weight that has barely moved | Small → allowed |
| Unimportant weight (low F) regardless of drift | Small → allowed |

---

## What you will see in the output

For each task:

| Metric | Meaning |
|--------|---------|
| **Task loss** | BCE loss on current task data at the final training epoch. |
| **EWC loss (raw)** | Σ_t Σ_i F_{t,i}(θ_i − θ*_{t,i})² before multiplying by λ. Grows as tasks accumulate. |
| **λ × EWC loss** | The scaled penalty actually added to the total loss. |
| **Ratio** | (λ × EWC) / task_loss. If >> 1: EWC dominates, model is effectively frozen. |
| **AvgPR** | PR-AUC on the *current task* after training — measures **plasticity** (can the model still learn?). |
| **ForgetPR** | PR-AUC on *Task 1* after training — measures **retention** (does the model still remember task 1?). |

A well-functioning EWC should show: `EWC ≥ Naive` on ForgetPR, `EWC ≈ Naive` on AvgPR.

In [6]:
# ═══════════════════════════════════════════════════════
#  CONFIGURE HERE
# ═══════════════════════════════════════════════════════
LAMBDA         = 50
EPOCHS         = 20
LR             = 0.01
FISHER_SAMPLES = 1000
DATASET        = "axis1_delta1.5_sudden.csv"
# ═══════════════════════════════════════════════════════

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import average_precision_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
with open(ROOT / 'config' / 'experiment.yaml') as f:
    cfg = yaml.safe_load(f)

from src.data.dataset import DataLoader as FraudDataLoader
from src.methods import FraudDetector
from src.methods.ewc import EWC
from src.utils import set_seed

loader  = FraudDataLoader(str(ROOT / cfg['paths']['base_dir'] / DATASET))
N_TASKS = 10

set_seed(42)
init_state = FraudDetector(input_features=8).state_dict()

print(f"Dataset: {DATASET}  |  λ={LAMBDA}  |  {EPOCHS} epochs")

In [ ]:
set_seed(42)

from src.methods.naive import train_on_task
from src.methods.ewc import EWC, train_with_ewc
from src.utils.metrics import test_on_task, compute_thesis_metrics

model_ewc = FraudDetector(input_features=8)
ewc       = EWC()
loss_info = []

pr_naive = np.full((N_TASKS, N_TASKS), np.nan)
pr_ewc   = np.full((N_TASKS, N_TASKS), np.nan)

for t in range(N_TASKS):
    X_tr, y_tr = loader.get_data_for_task(t, "train")

    # ── Naive: fresh model per task (same as run_experiments.py) ──
    set_seed(42)
    model_naive = FraudDetector(input_features=8)
    model_naive = train_on_task(model_naive, X_tr, y_tr, epochs=EPOCHS, lr=LR)

    # ── EWC: same model across tasks ──
    task_loss_before = nn.BCEWithLogitsLoss()(model_ewc(X_tr), y_tr).item()
    ewc_raw_before   = ewc.penalty(model_ewc).item()

    model_ewc = train_with_ewc(model_ewc, X_tr, y_tr, ewc,
                                importance_factor=LAMBDA, epochs=EPOCHS, lr=LR)
    ewc.update(model_ewc, (X_tr, y_tr), fisher_samples=FISHER_SAMPLES)

    # ── Loss info (after training) ──
    task_loss = nn.BCEWithLogitsLoss()(model_ewc(X_tr), y_tr).item()
    ewc_raw   = ewc.penalty(model_ewc).item()
    ewc_scaled = LAMBDA * ewc_raw

    loss_info.append({
        "task":       t + 1,
        "task_loss":  task_loss,
        "ewc_raw":    ewc_raw,
        "ewc_scaled": ewc_scaled,
        "ratio":      ewc_scaled / (task_loss + 1e-9),
    })

    # ── Evaluate on all tasks seen so far ──
    for j in range(t + 1):
        X_te, y_te = loader.get_data_for_task(j, "test")
        pr_naive[t, j] = test_on_task(model_naive, X_te, y_te)["pr_auc"]
        pr_ewc[t, j]   = test_on_task(model_ewc,   X_te, y_te)["pr_auc"]

In [ ]:
print(f"Table 1: Loss components (λ={LAMBDA})\n")
print(f"  {'Task':>4} | {'Task loss':>9} | {'EWC raw':>9} | {'λ x EWC':>9} | {'Ratio':>7}")
print(f"  {'─' * 48}")
for r in loss_info:
    if r["task"] == 1:
        print(f"  {r['task']:>4} | {r['task_loss']:>9.4f} | {'--':>9} | {'--':>9} | {'--':>7}")
    else:
        print(f"  {r['task']:>4} | {r['task_loss']:>9.4f} | {r['ewc_raw']:>9.6f} | {r['ewc_scaled']:>9.4f} | {r['ratio']:>6.2f}x")

In [ ]:
print(f"Table 2: PR-AUC & Forgetting per task — Naive vs EWC (λ={LAMBDA})\n")
print(f"  {'Task':>4} | {'Naive PR':>9} | {'EWC PR':>9} | {'Naive Fgt':>9} | {'EWC Fgt':>9}")
print(f"  {'─' * 52}")
for t in range(N_TASKS):
    pr_n = pr_naive[t, t]
    pr_e = pr_ewc[t, t]
    if t == N_TASKS - 1:
        print(f"  {t+1:>4} | {pr_n:>9.4f} | {pr_e:>9.4f} | {'--':>9} | {'--':>9}")
    else:
        fgt_n = pr_naive[N_TASKS - 1, t] - pr_naive[t, t]
        fgt_e = pr_ewc[N_TASKS - 1, t]  - pr_ewc[t, t]
        print(f"  {t+1:>4} | {pr_n:>9.4f} | {pr_e:>9.4f} | {fgt_n:>+9.4f} | {fgt_e:>+9.4f}")

_, _, avg_pr_n, fgt_pr_n, _ = compute_thesis_metrics(np.zeros_like(pr_naive), pr_naive)
_, _, avg_pr_e, fgt_pr_e, _ = compute_thesis_metrics(np.zeros_like(pr_ewc),   pr_ewc)
print(f"  {'─' * 52}")
print(f"  {'Avg':>4} | {avg_pr_n:>9.4f} | {avg_pr_e:>9.4f} | {fgt_pr_n:>+9.4f} | {fgt_pr_e:>+9.4f}")